# ================================================================
#                 YOLO MASTER TRAINING SYSTEM
#       Training from Scratch | Fine-Tuning | Validation
#                    Google Colab + Drive
# ================================================================
**Project**: `SVG` (ID: `34ab26f3-2778-4618-a7d4-cfae4bf561da`)

**Dataset Version**: `v1.0`

This notebook trains your YOLO model on free Google Colab GPUs (T4 / A100), mounts Google Drive for automatic checkpoint persistence, and exports validation metrics and best weights.

In [ ]:
#@title 1. Install Dependencies & GPU Verification
import sys
import subprocess

print("Installing latest Ultralytics & PyYAML...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics", "PyYAML", "requests"
])

import torch
print("=" * 65)
print("              YOLO MASTER TRAINING SYSTEM")
print("=" * 65)

if torch.cuda.is_available():
    print("[OK] GPU Device:", torch.cuda.get_device_name(0))
    print("[OK] CUDA Version:", torch.version.cuda)
    print("[OK] VRAM Available:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("[WARNING] GPU unavailable! Go to Runtime -> Change runtime type -> T4 GPU.")


In [ ]:
#@title 2. Connect Google Drive
from pathlib import Path
import shutil
import json
import time
from google.colab import drive

print("Mounting Google Drive...")
drive.mount("/content/drive")

BASE = Path("/content/yolo_master")
UPLOADS = BASE / "uploads"
DATASET_DIR = BASE / "dataset"
RUNS = BASE / "runs"
DRIVE_ROOT = Path("/content/drive/MyDrive/YOLO_MASTER_TRAINING")

for p in [UPLOADS, DATASET_DIR, RUNS, DRIVE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("[OK] Google Drive connected successfully.")
print("[OK] Checkpoints directory:", DRIVE_ROOT)


In [ ]:
#@title 3. Configuration & Hyperparameters
PROJECT_NAME = "SVG" #@param {type:"string"}
TRAINING_MODE = "1" #@param ["1", "2"] {type:"string"}
# Mode 1 = Train from scratch / pretrained base weights
# Mode 2 = Fine-tune an existing trained .pt checkpoint

BASE_MODEL = "yolo26l.pt" #@param ["yolo26l.pt", "yolov8n.pt", "yolov8s.pt", "yolov8m.pt", "yolov8x.pt", "custom"] {type:"string"}
CUSTOM_MODEL_NAME = "" #@param {type:"string"}

EPOCHS = 50 #@param {type:"integer"}
BATCH_SIZE = 4 #@param {type:"integer"}
IMAGE_SIZE = 640 #@param {type:"integer"}
OPTIMIZER = "AdamW" #@param ["AdamW", "SGD", "Adam", "auto"] {type:"string"}
LEARNING_RATE = 0.001 #@param {type:"number"}
FINAL_LR_FACTOR = 0.01 #@param {type:"number"}
WEIGHT_DECAY = 0.0005 #@param {type:"number"}
WARMUP_EPOCHS = 3 #@param {type:"integer"}
PATIENCE = 15 #@param {type:"integer"}
SAVE_PERIOD = 5 #@param {type:"integer"}
FREEZE_LAYERS = 0 #@param {type:"integer"}
RUN_VALIDATION = True #@param {type:"boolean"}

# Optional: Direct public download URL for dataset ZIP (e.g. Dropbox / GDrive / Cloud storage)
DIRECT_DATASET_URL = "" #@param {type:"string"}

print(f"Configured: Model={BASE_MODEL}, Epochs={EPOCHS}, Batch={BATCH_SIZE}, ImgSize={IMAGE_SIZE}")


In [ ]:
#@title 4. Acquire & Extract Dataset
import zipfile
import shutil
from google.colab import files

dataset_zip = None

# Method 1: Check if dataset zip is in Google Drive (Fastest)
drive_zips = list(DRIVE_ROOT.glob("*.zip"))
if drive_zips:
    dataset_zip = drive_zips[0]
    print(f"[FOUND] Found dataset in Google Drive: {dataset_zip.name}")

# Method 2: Check if dataset zip is in /content
if not dataset_zip:
    local_zips = [p for p in Path("/content").glob("*.zip") if not p.name.startswith(".")]
    if local_zips:
        dataset_zip = local_zips[0]
        print(f"[FOUND] Found uploaded dataset in /content: {dataset_zip.name}")

# Method 3: Download via DIRECT_DATASET_URL if specified
if not dataset_zip and DIRECT_DATASET_URL.strip():
    print(f"[DOWNLOADING] Fetching dataset from URL...")
    import subprocess
    target_zip = BASE / "dataset.zip"
    subprocess.call(["wget", "-q", "-O", str(target_zip), DIRECT_DATASET_URL.strip()])
    if target_zip.exists() and target_zip.stat().st_size > 1000:
        dataset_zip = target_zip
        print("[OK] Downloaded dataset successfully.")

# Method 4: Prompt file upload if no zip is located
if not dataset_zip or not dataset_zip.exists():
    print("[UPLOAD] No dataset zip found in Drive or /content.")
    print("Please upload your dataset ZIP file (downloaded from VisionForge Studio):")
    uploaded = files.upload()
    zip_files = [f for f in uploaded if f.lower().endswith(".zip")]
    if not zip_files:
        raise ValueError("Please upload your dataset as a .zip file.")
    dataset_zip = Path(zip_files[0])

# Safe extraction
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print(f"Extracting {dataset_zip.name}...")
with zipfile.ZipFile(dataset_zip, "r") as z:
    for member in z.infolist():
        destination = (DATASET_DIR / member.filename).resolve()
        if not destination.is_relative_to(DATASET_DIR.resolve()):
            raise ValueError("Unsafe ZIP path detected.")
    z.extractall(DATASET_DIR)

print("[OK] Dataset extracted to:", DATASET_DIR)


In [ ]:
#@title 5. Read & Validate data.yaml
import yaml

yaml_files = list(DATASET_DIR.rglob("data.yaml"))
if not yaml_files:
    yaml_files = list(DATASET_DIR.rglob("dataset.yaml"))

if not yaml_files:
    raise FileNotFoundError("data.yaml not found in dataset ZIP.")

SOURCE_YAML = yaml_files[0].resolve()
config = yaml.safe_load(SOURCE_YAML.read_text())

names = config.get("names")
if isinstance(names, list):
    dataset_classes = {i: str(n) for i, n in enumerate(names)}
elif isinstance(names, dict):
    dataset_classes = {int(k): str(v) for k, v in names.items()}
else:
    raise ValueError("Invalid class names in data.yaml.")

# Resolve relative dataset paths for Ultralytics
root = SOURCE_YAML.parent
config["path"] = str(root)

for key in ["train", "val", "test"]:
    value = config.get(key)
    if isinstance(value, str):
        p = Path(value)
        config[key] = str(p if p.is_absolute() else (root / p).resolve())
    elif isinstance(value, list):
        config[key] = [str(Path(x) if Path(x).is_absolute() else (root / x).resolve()) for x in value]

DATA_YAML = BASE / "master_data.yaml"
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))

print("[OK] CLASS VERIFICATION PASSED:", dataset_classes)
print("[OK] Master dataset YAML ready at:", DATA_YAML)


In [ ]:
#@title 6. Initialize YOLO Model
from ultralytics import YOLO

active_model_name = CUSTOM_MODEL_NAME.strip() if BASE_MODEL == "custom" and CUSTOM_MODEL_NAME.strip() else BASE_MODEL
source_model_path = None

if TRAINING_MODE == "1":
    print(f"Loading base model weights: {active_model_name}...")
    model = YOLO(active_model_name)
else:
    # Mode 2: Fine-tune existing checkpoint
    pt_files = list(DRIVE_ROOT.glob("*.pt")) + [p for p in Path("/content").glob("*.pt")]
    if pt_files:
        source_model_path = pt_files[0]
        print(f"[FOUND] Loading checkpoint from: {source_model_path.name}")
        model = YOLO(str(source_model_path))
    else:
        print("Please upload your starting .pt checkpoint:")
        up = files.upload()
        pts = [f for f in up if f.lower().endswith(".pt")]
        if not pts:
            raise ValueError("No .pt file uploaded for fine-tuning.")
        source_model_path = UPLOADS / ("starting_model_" + Path(pts[0]).name)
        shutil.copy2(pts[0], source_model_path)
        model = YOLO(str(source_model_path))

print("[OK] Model loaded successfully.")
print("[OK] Model classes:", model.names)


In [ ]:
#@title 7. Setup Drive Persistence & Checkpointing Callback
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"{PROJECT_NAME}_{TIMESTAMP}"
DRIVE_RUN = DRIVE_ROOT / RUN_NAME
DRIVE_RUN.mkdir(parents=True, exist_ok=True)

def save_checkpoint(trainer):
    epoch = trainer.epoch + 1
    if epoch % SAVE_PERIOD == 0 or epoch == trainer.epochs:
        weights = Path(trainer.save_dir) / "weights"
        for filename in ["last.pt", "best.pt"]:
            src = weights / filename
            if src.exists():
                dst = DRIVE_RUN / f"epoch_{epoch:03d}_{filename}"
                shutil.copy2(src, dst)
                print(f"[CHECKPOINT] Saved to Google Drive: {dst.name}")

model.add_callback("on_model_save", save_checkpoint)
print("[OK] Checkpoints configured to auto-save every", SAVE_PERIOD, "epochs.")
print("[OK] Google Drive destination:", DRIVE_RUN)


In [ ]:
#@title 8. Start YOLO Master Training
print("=" * 65)
print("             STARTING YOLO MASTER TRAINING")
print("=" * 65)
print("Project:", PROJECT_NAME)
print("Epochs:", EPOCHS)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer:", OPTIMIZER)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=2,
    optimizer=OPTIMIZER,
    lr0=LEARNING_RATE,
    lrf=FINAL_LR_FACTOR,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    patience=PATIENCE,
    freeze=FREEZE_LAYERS,
    mosaic=1.0,
    mixup=0.0,
    degrees=0.0,
    flipud=0.0,
    fliplr=0.5,
    seed=42,
    deterministic=True,
    device=0 if torch.cuda.is_available() else "cpu",
    save=True,
    save_period=SAVE_PERIOD,
    plots=True,
    project=str(RUNS),
    name=RUN_NAME,
    exist_ok=False,
    verbose=True
)

RUN_DIR = Path(results.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
LAST = RUN_DIR / "weights" / "last.pt"

FINAL_BEST = DRIVE_RUN / "best.pt"
if BEST.exists():
    shutil.copy2(BEST, FINAL_BEST)
if LAST.exists():
    shutil.copy2(LAST, DRIVE_RUN / "last.pt")

for item in RUN_DIR.iterdir():
    if item.is_file():
        shutil.copy2(item, DRIVE_RUN / item.name)

print("[OK] Training complete! Best weights saved to Google Drive:", FINAL_BEST)


In [ ]:
#@title 9. Validation & Metrics Reporting
metrics = {}
if RUN_VALIDATION:
    print("Running final validation on best weights...")
    trained_model = YOLO(str(BEST))
    validation = trained_model.val(
        data=str(DATA_YAML),
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        plots=True,
        project=str(RUNS),
        name=RUN_NAME + "_validation"
    )
    metrics = {
        "precision": float(validation.box.mp),
        "recall": float(validation.box.mr),
        "mAP50": float(validation.box.map50),
        "mAP50_95": float(validation.box.map)
    }
    print("=" * 45)
    print("            VALIDATION RESULTS")
    print("=" * 45)
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    (DRIVE_RUN / "validation_metrics.json").write_text(json.dumps(metrics, indent=2))
    print("[OK] Metrics saved to:", DRIVE_RUN / "validation_metrics.json")


In [ ]:
#@title 10. Summary & Completion Report
print("=" * 65)
print("                TRAINING FULLY COMPLETED")
print("=" * 65)
print("Best model weights in Drive:", FINAL_BEST)
print("Google Drive Folder:", DRIVE_RUN)
if metrics:
    print("Final mAP50:", f"{metrics.get('mAP50', 0.0):.4f}")
    print("Final Precision:", f"{metrics.get('precision', 0.0):.4f}")
    print("Final Recall:", f"{metrics.get('recall', 0.0):.4f}")

print("\nAll model checkpoints, plots, and metrics are preserved in Google Drive.")
print("You can import 'best.pt' directly into VisionForge Model Registry.")
print("=" * 65)
